##1.
####The main problem was understanding how the Google Cloud project, notebooks here, and billing credits work together. Unlike regular Colab, I needed to make sure the notebook used the managed cloud environment and honestly took some time to even find how to use notebooks. I also had to make sure to stop the runtime when finished. The billing balance did not update immediately after I finished, which made it harder to check credit consumption and add a proof right away. Once the environment was set up and all ready, the Pandas and scikit-learn workflow was similar to regular Colab. But to be honest this environment seems to be too complex to work in, at least compared to Colab.

##2.
####I identified missing measurements by finding -200 values and replacing them with nan. There were 1,683 missing CO measurements and 8,443 missing NMHC measurements. I removed rows with missing CO because CO was my target, and used SimpleImputer with the median for numerical inputs. At first, old target variables still included -200, producing an RMSE of about 53. After rebuilding the targets from the cleaned data, Linear Regression’s RMSE became 0.4694. Median imputation let the models use not fully complete feature rows without treating missing measurements as real concentrations.

##3.
####I used ColumnTransformer to applty median imputation and scaling to numerical features. categorical features received most-frequent imputation and one-hot encoding. Pipeline connected these two steps to the model, so preparation and training happened together. I split the data before fitting the pipeline. This meant medians, scaling values, and encoding categories were learned from the training data and then used again on the test data. The test data did not impact those preprocessing values. I also removed CO(GT) from the inputs because it was the target.

##4.
####StandardScaler generally gave better prediction results. MinMaxScaler trained faster in the recorded runs. For Softmax, StandardScaler achieved 87.62% accuracy in 86 iterations, compared with 86.97% in 68 iterations using MinMaxScaler. For SGD with log loss, accuracy was 87.82% with StandardScaler and 85.60% with MinMaxScaler. Both scalers put numerical features on comparable scales, it helped optimization avoid being dominated by large sensor values. StandardScaler gave better accuracy and lower log loss here, although it required more iterations.

##5.
####I did SGDClassifier using the loss function, regularization parameter alpha, learning-rate setting, maximum iterations, and stopping tolerance. SGD updates its weights using individual training examples, it makes it useful for large datasets. A closed-form least-squares method solved for coefficients directly. The Logistic Regression model in this project used lbfgs, which is also iterative rather than closed-form. With log_loss, SGD provided class probabilities. These allowed log-loss evaluation, which harms confident incorrect predictions. Accuracy measured overall correctness, while macro F1 gave each category equal importance. StandardScaler with SGD log loss achieved 87.82% accuracy and 0.8735 macro F1. Hinge loss did not provide probability outputs, so its log-loss result was left blank.

##6.
####Increasing Ridge’s alpha reduced the coefficient norm from about 1.7980 at alpha 0.01 to 1.6157 at alpha 10. Its RMSE stayed close to 0.4694, and no coefficients became zero. Ridge  reduced coefficient sizes while retaining all features.
####Lasso removed features more strongly. At alpha 0.01, it set 30 coefficients to zero and had an RMSE of 0.5012. At alpha 0.1, it set 43 to zero and RMSE increased to 0.5534. At alpha 10, all coefficients were zero, showing excessive regularization and underfitting.
####At alpha 1, Lasso retained only the benzene concentration feature, suggesting it contained useful information for predicting CO. But, retaining additional measurements gave better predictions. These relationships are predictive associations, not proof that one pollutant causes another.